# Recurrent networks

**Lecture 16** · Géron, Chapter 13

Applications of Machine Learning — BSc Mathematics of Artificial Intelligence

---

**How to use this notebook.** Read before you run. Every code cell is preceded
by the specification that would produce it — input, output, constraint, check.

Section 7 measures two failure conditions beside the protocol the lecture
reports — the same recurrent model under a random split, and the 2020 regime
change — and prices each against the naive baseline on the same rows. Nothing in
this notebook is wrong on purpose.

Runs on CPU. Nothing here needs an accelerator.

**A CPU runtime is enough.** The largest model here has a few thousand
parameters. If Colab offers you a GPU, decline it — for sequences this short the
transfers cost more than the arithmetic saves.

**Section 7** runs the whole
experimental programme the slides report — the six-model ladder, the same model
under a random split, fourteen horizons, and the 2020 regime change — under one
protocol. It is the lecture's result rather than a
detour, which is why it is run here rather than quoted.

**About the specification boxes.** Every code cell in this notebook is preceded
by a box naming four things: the input, the output, the constraint the method
must respect, and a check whose answer you can work out before running
anything. Read the box, answer the check in your head, then run the cell.

Each box then shows two ways of asking an assistant for the cell: the **novice
prompt**, which is what most people would type, and the **pro prompt**, which
names every decision the cell depends on — followed by **why the pro prompt
works better**. That last field is tested, not guessed: each novice prompt was
given to a current AI assistant together with this notebook up to that cell,
and the why reports what came back. Often the assistant gets much of the cell
right unasked; the why then says what it still got wrong, or did without being
required to, and which of the pro's choices settles it. The two prompts are
written for this cell, not recorded: nobody claims either produced the code
below. Neither states the check's answer, because the check is how *you* find
out whether the answer is wrong.


## 1 · Setup

> **Specification**
>
> **output** · library versions printed, one seed set everywhere, plot defaults fixed once
>
> **constraint** · not examinable — engineering hygiene, kept out of the way of the argument
>
> ---
>
> **novice prompt** · “set up the notebook with PyTorch”
>
> **pro prompt** · “Before importing torch, set OMP_NUM_THREADS to 4 and KMP_DUPLICATE_LIB_OK to TRUE with `os.environ.setdefault`. Import numpy, pandas, torch, torch.nn and matplotlib; print the Python, pandas and torch versions; seed numpy and torch with `RANDOM_STATE = 42`; and set the plot defaults once: figure size (11, 3.2), a light grid, font size 11.”
>
> **why the pro prompt works better** · 'Set up the notebook with PyTorch' says nothing about import order, and asked this way an assistant typically imports torch first and calls `torch.set_num_threads` afterwards, which cannot prevent the silent OpenMP deadlock some machines hit once training starts. Without the surrounding prose it does not reliably import pandas or print the Python and pandas versions, and it picks its own figure size and grid. It usually seeds numpy and torch unasked, but nothing in the request requires it. What does the work is the pro's `os.environ.setdefault` calls for OMP_NUM_THREADS and KMP_DUPLICATE_LIB_OK placed before the torch import.
>
> ---
>
> **try** · change the seed here and re-run the ladder in Section 7. The two univariate RNNs can move by tens of thousands of boardings (the ladder prints them under four seeds). Hold that spread in mind when the ladder reports the differences between six architectures.

In [ ]:
# Not examinable, and only needed on some machines: PyTorch, numpy and
# torchvision can each end up loading their own OpenMP runtime, and with more
# than one loaded a training cell can deadlock -- no error, no output, and no
# CPU use. These have to be set BEFORE torch is imported, because they are read
# at import time and after that they do nothing.
import os
os.environ.setdefault("OMP_NUM_THREADS", "4")
os.environ.setdefault("KMP_DUPLICATE_LIB_OK", "TRUE")

# --- setup -------------------------------------------------------------------
# Not examinable: this is engineering hygiene, not machine learning.
import sys
from pathlib import Path
import tarfile, urllib.request

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import matplotlib.pyplot as plt

print(f"python  {sys.version.split()[0]}")
print(f"pandas  {pd.__version__}")
print(f"torch   {torch.__version__}")

RANDOM_STATE = 42                  # one seed, used everywhere
np.random.seed(RANDOM_STATE)
torch.manual_seed(RANDOM_STATE)

# A time series is plotted more often than anything else in this lecture, so it
# is worth setting the defaults once.
plt.rcParams.update({"figure.figsize": (11, 3.2), "axes.grid": True,
                     "grid.alpha": 0.3, "font.size": 11})

## 2 · The data

The Chicago Transit Authority publishes one row per day: how many people
boarded a bus, how many boarded a train, and what kind of day it was.

> **Specification**
>
> **input** · the CTA daily-boardings CSV, fetched once and cached
>
> **output** · one row a day: date, day type, bus and rail counts
>
> **constraint** · download only if the file is not already on disk, so the notebook is re-runnable offline
>
> **check** · print the row count and compare it with what the publisher states
>
> ---
>
> **novice prompt** · “load the CTA ridership data”
>
> **pro prompt** · “Write `load_ridership()` that downloads ageron's `ridership.tgz` into `datasets/` only if it is absent, extracts it with `filter='data'`, reads the CTA CSV with `service_date` parsed as dates into `raw`, and prints the published row count.”
>
> **why the pro prompt works better** · 'Load the CTA ridership data' mostly gets the plumbing right. Asked this way, an assistant typically fetches ageron's `ridership.tgz` only if absent, extracts with `filter='data'` and parses `service_date`, though the request requires none of it. What it does not reliably do is stop at loading. It typically renames columns, drops `total` and calls `drop_duplicates()` after the date has become the index, so rows are compared without their dates. It then prints only the cleaned count, and the file as published is never seen. The pro's separate `raw` frame and 'prints the published row count' keep that state visible, so drift or rows lost to cleaning show up.
>
> ---
>
> **try** · re-download the CSV and compare the printed row count with the one printed here today. Every score below is tied to these days, and this is the one cell where that could quietly stop being true without anything raising.

In [ ]:
# --- the data ----------------------------------------------------------------
def load_ridership():
    """Daily boarding totals for the Chicago Transit Authority."""
    tarball = Path("datasets/ridership.tgz")
    if not tarball.is_file():
        Path("datasets").mkdir(parents=True, exist_ok=True)
        url = "https://github.com/ageron/data/raw/main/ridership.tgz"
        urllib.request.urlretrieve(url, tarball)
        with tarfile.open(tarball) as t:
            t.extractall(path="datasets", filter="data")
    return pd.read_csv(
        "datasets/ridership/CTA_-_Ridership_-_Daily_Boarding_Totals.csv",
        parse_dates=["service_date"])

raw = load_ridership()
print(f"{len(raw):,} rows as published")

### Tidying, and one line that matters

Four operations. Three are housekeeping and one is a decision:

* rename the columns, because `service_date` is long and we will type it often
* sort by date and index by it — a series with an unsorted index is a trap
* drop `total`, which is exactly `bus + rail` and so carries nothing
* **drop duplicate rows.** The published file repeats some days.

Count the duplicates rather than assuming there are none.

> **Specification**
>
> **input** · the raw frame, one row a day, columns as published
>
> **output** · a frame indexed by date with bus and rail only
>
> **constraint** · drop `total` — it is exactly bus + rail — and drop duplicate rows, reporting how many
>
> **check** · print the count removed and the date range that survives
>
> ---
>
> **novice prompt** · “clean up the dataframe”
>
> **pro prompt** · “Copy `raw` to `df`, rename the columns to date, day_type, bus, rail, total, sort by date and make it the index, drop `total`, then `drop_duplicates()` and print how many rows were removed and the date range that remains.”
>
> **why the pro prompt works better** · 'Clean up the dataframe' usually gets a sound cleaning unasked: a current assistant typically copies `raw`, renames the columns, sorts and indexes by date, checks `total` equals bus plus rail before dropping it, removes repeated rows and prints the count and date range. But the request requires none of it, so nothing guarantees or checks it. A missed duplicate is a re-weighting rather than a leak: that day counts twice in every network's training loss and in the baselines, nudging every MAE silently. The pro makes each step a stated requirement, ending on an explicit `drop_duplicates()` with the removed-row count and surviving date range printed.
>
> ---
>
> **try** · keep the duplicate rows and re-run the notebook. Every MAE moves a little and no assert fires. Duplicates in a time series are not a leak, they are a re-weighting: some days now count twice in the fit and once in the score.

In [ ]:
df = raw.copy()
df.columns = ["date", "day_type", "bus", "rail", "total"]
df = df.sort_values("date").set_index("date")
df = df.drop("total", axis=1)          # it is just bus + rail

before = len(df)
df = df.drop_duplicates()
print(f"{before - len(df)} duplicate rows removed, {len(df):,} days left")
print(f"from {df.index.min().date()} to {df.index.max().date()}")
df.head(3)

### `day_type`

Three values: `W` for a weekday, `A` for a Saturday, `U` for a Sunday **or a
public holiday**. That last word is why the column is worth having — the model
cannot see a calendar, and the fourth of July behaves like a Sunday whatever the
date says.

> **Specification**
>
> **input** · the tidied frame
>
> **output** · how many days of each type, and the three days around Memorial Day 2019
>
> **constraint** · do not assume U means Sunday — show a case where it is a public holiday on a Monday
>
> **check** · the three printed values are A, U, U
>
> ---
>
> **novice prompt** · “what values does day_type take?”
>
> **pro prompt** · “Print `df['day_type'].value_counts()`, then the `day_type` values for 2019-05-25 to 2019-05-27: the weekend before Memorial Day and the holiday itself.”
>
> **why the pro prompt works better** · 'What values does day_type take?' gets a sound answer: asked this way, an assistant prints `value_counts()` unasked and usually goes further, cross-tabulating the codes against weekday names, which shows in aggregate that some weekdays are coded U. What it never does is point at one dated case, so the claim that holidays share Sunday's code stays a count buried in a table. This column becomes a model input in Section 6, so the pro fixes the window 2019-05-25 to 2019-05-27, the weekend before Memorial Day and the holiday itself, so the output shows a Monday holiday coded U.
>
> ---
>
> **try** · count the U days that fall on a Monday to Friday. Those are the public holidays, and they are the only reason this column beats a plain day-of-week feature. How many are in the pool, and is that enough to learn anything from?

In [ ]:
print(df["day_type"].value_counts().to_string())

# Memorial Day 2019 fell on a Monday, and the data knows it
print(list(df.loc["2019-05-25":"2019-05-27"]["day_type"]))

## 3 · The series, the baselines, and an honest protocol

Before any network: look at the series at two time scales, fix the days we work
on, choose a metric, measure what forecasts with no model score, and fit a
linear model on 56 lags — first under the shuffled split used for independent
rows, then forward in time, as it would be used.

Twenty-one years, then ten weeks. Each shows a fact the other hides.

> **Specification**
>
> **input** · the rail series, all twenty-one years
>
> **output** · one line plot, rail boardings against date
>
> **constraint** · thin line, no markers — the shape is the point, not the individual days
>
> **check** · the 2020 cliff is visible without being pointed at
>
> ---
>
> **novice prompt** · “plot rail ridership over time”
>
> **pro prompt** · “Plot `df['rail']` over its full date range as one thin line (lw 0.6, no markers), with the y-axis labelled rail boardings.”
>
> **why the pro prompt works better** · 'Plot rail ridership over time' leaves the shape of the figure open. Asked this way, an assistant does plot the full span of raw daily values as a thin, unmarked line, but with the notebook's prose present it tends to grow the request into a multi-panel figure. It adds a few weeks shown with markers and a single year before 2020, so the full series becomes one view among several and the 2020 break that ends the process stops being the point. It also labels the axis its own way, rescaled to millions. Asking for `df['rail']` over its full date range as 'one thin line', with the y-axis labelled rail boardings, keeps the single view this notebook discusses.
>
> ---
>
> **try** · restrict the plot to 2016 onward. The slow decline through the 2010s vanishes and the cliff becomes the whole story. Both plots are honest; they answer different questions, and only one of them is the question this notebook asks.

In [ ]:
fig, ax = plt.subplots(figsize=(11, 3.2))
df["rail"].plot(ax=ax, lw=0.6, color="#0b3d62")
ax.set_ylabel("rail boardings")
ax.set_title("Twenty-one years — and one event that ends the series we know")
plt.show()

**What the plot says.** A slow decline through the 2010s, and then in March 2020
a cliff. Everything after that cliff is a different process. We are not going to
model across it: this notebook works on **2016-01 to 2019-05**, and returns to
the cliff in Section 7 to ask what a model owes you when the world changes.

> **Specification**
>
> **input** · bus and rail over ten weeks of 2019
>
> **output** · both series on one axis, with markers
>
> **constraint** · a window short enough that individual days are distinguishable
>
> **check** · every week shows a weekend trough; some weeks show a third
>
> ---
>
> **novice prompt** · “zoom in on a few weeks of ridership”
>
> **pro prompt** · “Plot `bus` and `rail` from `df` for 2019-03 to 2019-05 on one axis with small point markers, so that individual days can be told apart.”
>
> **why the pro prompt works better** · 'Zoom in on a few weeks of ridership' leaves the window to the assistant, and asked this way it does not reliably choose the same one. Sometimes it covers all of spring, sometimes it starts weeks later or stops well short of the end of May. A window that stops early can miss the late-May public holiday, and then the third dip that `day_type` encodes is not in the figure. An assistant typically plots `bus` and `rail` on one axis with small dot markers unasked, but the request does not require it. The pro's fixed window, 2019-03 to 2019-05, makes the figure the one this notebook discusses.
>
> ---
>
> **try** · widen this window to a full year, keeping the markers. The weekly trough is still there and is no longer legible. A figure's window is an argument about what you want the reader to be able to see.

In [ ]:
fig, ax = plt.subplots(figsize=(11, 3.2))
df.loc["2019-03":"2019-05", ["bus", "rail"]].plot(ax=ax, marker=".", ms=4)
ax.set_ylabel("boardings")
ax.set_title("Ten weeks — the shape that actually matters")
plt.show()

**Weekly seasonality, and a second trough.** Every week has a deep weekend
trough. Some weeks have a third dip in the middle, and those are the public
holidays — which is exactly what `day_type` marks and what a bare day-of-week
feature would miss.

Whatever we build has to know what day of the week it is. The cheapest way to
tell it is to show it the last week, or the last eight.

> **Specification · the window we will work on, fixed before any model**
>
> **input** · the tidied frame
>
> **output** · the rail pool and the days that can be scored
>
> **constraint** · fix the span and the window length HERE, before any baseline — a window chosen after seeing a result is a hyperparameter fitted on the test set
>
> **check** · assert the first WINDOW days are excluded from the targets. They are consumed by the history every model needs, so they cannot be scored.
>
> ---
>
> **novice prompt** · “pick the training period and the window size”
>
> **pro prompt** · “Set `WINDOW = 56`, `pool = df['rail']['2016-01':'2019-05']` and `target = pool[WINDOW:]`. Assert the target has `len(pool) - WINDOW` days, and print the pool's date range and the first scored day.”
>
> **why the pro prompt works better** · 'Pick the training period and the window size' usually gets sensible constants: asked this way, an assistant fixes a 56-day window and a 2016-01 to 2019-05 pool, ending before 2020, without searching, though the request does not require that. What it does not reliably do is define one scored set. It typically splits the pool into train, validation and test blocks and windows each separately, so each block loses its own first days and later baselines are scored on different, smaller sets. It does not print the first scored day. The pro sets `target = pool[WINDOW:]` and asserts it has `len(pool) - WINDOW` days.
>
> ---
>
> **try** · set WINDOW = 14 and re-run everything below. More days become scoreable and every MAE moves. Nothing here stops you trying 14, 28 and 56 and keeping the best — except that you would then have to say which days you chose it on.

In [ ]:
# The pool we will work on, and the days we will be scored on. The first 56 days
# are consumed by the window the models need, so they cannot be targets.
WINDOW = 56
pool = df["rail"]["2016-01":"2019-05"]
target = pool[WINDOW:]

assert len(target) == len(pool) - WINDOW
print(f"pool   {len(pool):,} days, {pool.index.min().date()} to {pool.index.max().date()}")
print(f"scored {len(target):,} days, from {target.index.min().date()}")

### Choose a metric, and the baseline to beat

Before a single model. Three candidates:

| Metric | What it says | Why not |
|---|---|---|
| **MAE** | average error in boardings | — |
| RMSE | punishes large errors more | one strike day dominates the score |
| MAPE | average error as a percentage | division by a small denominator |

**The MAPE trap.** MAPE divides by the truth. Christmas Day has about a sixth of
a Tuesday's ridership, so a 20,000-boarding error on Christmas counts for as much
as a 120,000-boarding error on a Tuesday. The metric would spend the model's
capacity on the days nobody is planning staffing for.

We use **MAE, in boardings**. It is in the units the operations team already
thinks in, and one unit of it is one person.

> **Specification**
>
> **input** · two aligned series, either of which may have gaps
>
> **output** · mean absolute error, in boardings
>
> **constraint** · ignore days where either side is missing rather than filling them
>
> **check** · mae(x, x) is 0, and a series against a shifted copy of itself is not
>
> ---
>
> **novice prompt** · “write a mean absolute error function”
>
> **pro prompt** · “Write `mae(truth, forecast)` that converts both to pandas Series, masks out days where either is NaN, and returns the mean absolute difference as a float in boardings. Print `mae(pool, pool)` and `mae(pool, pool.shift(7))`.”
>
> **why the pro prompt works better** · 'Write a mean absolute error function' says nothing about missing values, yet every baseline below is a shifted series with NaN at its start. Asked this way, an assistant typically writes a careful function that raises on any NaN. As a result `mae(pool, pool.shift(7))` crashes, and shifted baselines must be trimmed by hand first. It also tends to work on plain arrays rather than Series, and it does not reliably return a plain float. It usually adds a zero-error check unasked, though on toy data. The printed shifted-baseline test catches this at once. The pro's explicit mask on days where either side is NaN decides which days are scored.
>
> ---
>
> **try** · delete the mask and subtract the two series directly. It still returns a number, because pandas skips NaN in mean(). Now do the same after .to_numpy() and watch it become nan. Which of those two failures would you rather have in a notebook you are not reading closely?

In [ ]:
def mae(truth, forecast):
    """Mean absolute error, ignoring days where either side is missing."""
    truth, forecast = pd.Series(truth), pd.Series(forecast)
    mask = truth.notna() & forecast.notna()
    return float((truth[mask] - forecast[mask]).abs().mean())

print(f"mae against itself      {mae(pool, pool):.1f}")
print(f"mae against a shift     {mae(pool, pool.shift(7)):,.0f} boardings")

### The baselines

A number is only good or bad next to another number. Three forecasts that
require no fitting at all:

> **Specification**
>
> **input** · the 2016-01 to 2019-05 rail pool
>
> **output** · MAE for four forecasts that require no fitting, and the MAPE of copying last week
>
> **constraint** · every forecast scored on exactly the same days, so they are comparable
>
> **check** · copying last week beats copying yesterday — if not, the alignment is wrong
>
> ---
>
> **novice prompt** · “compare some simple baseline forecasts”
>
> **pro prompt** · “On `target`, score four forecasts with `mae`: a constant equal to the mean of `pool` up to 2018-12, `pool.shift(1)`, `pool.shift(7)` and `pool.shift(14)`, each sliced `[WINDOW:]` so all four are scored on exactly the same days. Print one MAE per row, then the mean absolute percentage error of `pool.shift(7)` on the same days and how many times smaller its MAE is than `pool.shift(1)`'s.”
>
> **why the pro prompt works better** · 'Compare some simple baseline forecasts' lets an assistant choose its own set. Asked this way, it typically drops the constant forecast and `pool.shift(14)` and adds same-weekday averages instead, so the most basic reference point is never computed. It does not reliably print a MAPE, or state how many times smaller last week's MAE is than yesterday's. It usually scores every forecast on the same days unasked, aligning each to `target` with an assert, but the request does not require it. The pro fixes the four forecasts and the closing `pool.shift(7)` comparison, starting from 'a constant equal to the mean of `pool` up to 2018-12'.
>
> ---
>
> **try** · add 'copy 28 days back' to the table: also a whole number of weeks, but four times as far away. Where does it land relative to the day-before and last-week rows, and what does that say about how much of the naive forecast is the weekday and how much is the recency?

In [ ]:
baselines = {
    "a constant (the mean)":  pd.Series(pool[:"2018-12"].mean(), index=target.index),
    "copy the day before":    pool.shift(1)[WINDOW:],
    "copy last week":         pool.shift(7)[WINDOW:],
    "copy two weeks back":    pool.shift(14)[WINDOW:],
}
for name, forecast in baselines.items():
    print(f"{name:24s} MAE {mae(target, forecast):>10,.0f}")

# the same naive error as a percentage of each day, over every scored day
naive = baselines["copy last week"]
print(f"\ncopy last week, MAPE over all {len(target):,} days: "
      f"{100 * ((target - naive).abs() / target).mean():.1f}%")
print(f"copying last week beats copying the day before by a factor of "
      f"{mae(target, baselines['copy the day before']) / mae(target, naive):.1f}")

**Read that table before going on.** Copying the same weekday one week ago is
about **2.4 times better** than copying yesterday, and roughly three times better
than the mean. It costs nothing, it has no parameters, it cannot overfit, and it
already knows about weekends. It knows nothing about public holidays: on a
weekday holiday it copies an ordinary weekday, and a week later it copies the
holiday back. Those are its worst days.

That is the number to beat. Not the mean. Not zero.

### The number to beat

Everything below is measured against it, including the parts that disappoint:
the linear model now, every network after it.

### A linear model on 56 lags, and a random split, measured

$\hat y_{t+1} = b + \sum_{k=0}^{55} w_k\, y_{t-k}$: fifty-six weights and a
bias. With $w_6 = 1$ and every other weight zero it is copying last week, so it
contains the baseline. First, a shuffled five-fold split with a fixed seed — the
protocol for independent rows.

> **Specification**
>
> **input** · the pool as 56-lag windows
>
> **output** · a cross-validated MAE from a shuffled five-fold split
>
> **constraint** · KFold(shuffle=True) with a fixed seed, exactly as it would be used on independent rows — this is the failure condition being measured
>
> **check** · look at the fold spread — they agree with each other, which is what a stable measurement of the wrong quantity looks like.
>
> ---
>
> **novice prompt** · “cross-validate the linear model”
>
> **pro prompt** · “Build 56-lag windows `Xw`, `yw` from `pool / 1e6` and score `LinearRegression` with `cross_val_score` under `KFold(5, shuffle=True, random_state=RANDOM_STATE)`, MAE converted to boardings, printing each fold. Then repeat over seeds 0 to 19 and print the mean and the spread across seeds as `RANDOM_CV`.”
>
> **why the pro prompt works better** · 'Cross-validate the linear model' does not name a protocol, and asked this way an assistant does not choose reliably. With this notebook's prose it runs shuffled `KFold` beside forward-chaining. From the code alone it switches to `TimeSeriesSplit` and never produces the shuffled number this cell exists to measure. It typically prints each fold unasked. It does not repeat the shuffle over seeds, so it credits the whole shuffled-versus-forward gap to the splitter when seed-to-seed variance could explain part of it. The pro fixes `KFold(5, shuffle=True, random_state=RANDOM_STATE)` and then requires seeds 0 to 19, printing the mean and spread as `RANDOM_CV`.
>
> ---
>
> **try** · raise the twenty seeds to a hundred and read the spread across seeds again. It barely moves: the shuffled protocol is stable, and it is stably measuring the wrong quantity. Stability is not validity.

In [ ]:
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import KFold, cross_val_score

values = pool.values / 1e6
Xw = np.stack([values[i:i + WINDOW] for i in range(len(values) - WINDOW)])
yw = values[WINDOW:]
print(f"Xw {Xw.shape}   yw {yw.shape}")

model = LinearRegression()
cv = KFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
folds_random = -cross_val_score(model, Xw, yw, cv=cv,
                                scoring="neg_mean_absolute_error") * 1e6
print(f"random 5-fold   MAE {folds_random.mean():>10,.0f}   "
      f"folds {np.round(folds_random).astype(int)}")

# One shuffled split is one draw. The slides quote the mean over twenty seeds,
# because a single seed of this splitter moves by more than the effect we are
# about to attribute to the splitter itself.
seed_means = np.array([
    -cross_val_score(model, Xw, yw, cv=KFold(5, shuffle=True, random_state=s),
                     scoring="neg_mean_absolute_error").mean() * 1e6
    for s in range(20)])
RANDOM_CV = float(seed_means.mean())
print(f"over 20 seeds   MAE {RANDOM_CV:>10,.0f}   "
      f"sd across seeds {seed_means.std():>7,.0f}")

> **Specification · the split a deployed system actually faces**
>
> **input** · the same windows, fit on everything up to the end of 2018
>
> **output** · MAE on the five months that follow, and the naive forecast on the same rows
>
> **constraint** · score the naive baseline on the SAME test rows — otherwise a gap between protocols could be the rows being harder rather than the protocol being honest
>
> **check** · report both, and their ratio. The ratio is a skill score, and it is the only thing comparable between two protocols that scored different days.
>
> ---
>
> **novice prompt** · “now test it on later data, like a real forecast”
>
> **pro prompt** · “Fit `LinearRegression` on windows whose target date is before 2019-01-01 and score MAE in boardings on the rest. Score copy-last-week, `Xw[:, -7]`, on the same test rows, and print both, their ratio as a skill score, and the shuffled `RANDOM_CV` beside copy-last-week on all the rows it scores. Print each protocol's margin over the baseline, how much harder the forward rows are for copy-last-week than all the rows, and by how much the random split overstates the skill.”
>
> **why the pro prompt works better** · 'Test it on later data' gets a forward evaluation, and a current assistant usually scores copy-last-week on the same test rows unasked. What it typically does is set the forward MAE beside the shuffled `RANDOM_CV` and report the raw ratio as how much shuffling flatters the model, though the rows differ and later days are harder even for copy-last-week. It never scores that baseline on all rows `RANDOM_CV` covers, only sometimes prints a skill ratio, and does not reliably make a single date cut. The pro's cut at 2019-01-01, with copy-last-week, `Xw[:, -7]`, scored on each protocol's own rows and each ratio printed as a skill score, compares protocols rather than days.
>
> ---
>
> **try** · move the cut from 2019-01-01 to 2018-07-01. Both MAEs change and the skill ratio changes much less, because the ratio divides out how hard the rows are. That is the reason to report it.

In [ ]:
dates = pool.index[WINDOW:]
tr = dates < "2019-01-01"

fit = LinearRegression().fit(Xw[tr], yw[tr])
HONEST_SPLIT = 1e6 * float(np.abs(fit.predict(Xw[~tr]) - yw[~tr]).mean())
HONEST_NAIVE = 1e6 * float(np.abs(Xw[~tr, -7] - yw[~tr]).mean())

print(f"train {tr.sum():,} days, test {(~tr).sum():,} days")
print(f"linear, forward from 2019-01-01  MAE {HONEST_SPLIT:>10,.0f}")
print(f"copy last week, same rows     MAE {HONEST_NAIVE:>10,.0f}")
print(f"skill (model / naive)             {HONEST_SPLIT / HONEST_NAIVE:>10.3f}")

# The shuffled folds score every one of the rows once, so ITS same-rows
# baseline is copy-last-week over all of them.
POOL_NAIVE = 1e6 * float(np.abs(Xw[:, -7] - yw).mean())
print(f"\nrandom split, same model      MAE {RANDOM_CV:>10,.0f}")
print(f"copy last week, all {len(yw):,} rows MAE {POOL_NAIVE:>10,.0f}")
print(f"skill (model / naive)             {RANDOM_CV / POOL_NAIVE:>10.3f}")

claimed = 1 - RANDOM_CV / POOL_NAIVE
real    = 1 - HONEST_SPLIT / HONEST_NAIVE
print(f"\nmargin over copying last week, random split   {claimed:6.1%}")
print(f"margin over copying last week, forward        {real:6.1%}")
print(f"forward MAE above the random-split MAE       {HONEST_SPLIT / RANDOM_CV - 1:+6.1%}")
print(f"share of the claimed margin the forward split removes  "
      f"{(claimed - real) / claimed:.0%}")
print(f"\nthe forward rows are harder before any model: copy-last-week "
      f"{HONEST_NAIVE / POOL_NAIVE - 1:+.0%}")
print(f"the random split overstates the skill by "
      f"{(HONEST_SPLIT / HONEST_NAIVE) / (RANDOM_CV / POOL_NAIVE) - 1:.1%}")

> **Specification · the matched random split**
>
> **input** · twenty random splits with the SAME training and test sizes as the rolling backtest
>
> **output** · the mean MAE
>
> **constraint** · match the sizes — TimeSeriesSplit's first fold trains on a fifth of the data, so a comparison against it confounds the protocol with the training-set size
>
> **check** · this is the row that isolates the protocol. Same model, same sizes, same metric; only the ordering differs.
>
> ---
>
> **novice prompt** · “compare a random split and a time split fairly”
>
> **pro prompt** · “With `L, T = 700, 98`, run a rolling-origin backtest from starts 700, 798, 896, 994 and 1092, each training on the preceding L windows and testing on the next T, and 20 `ShuffleSplit` draws with the same train and test sizes. Score `LinearRegression` and copy-last-week on every test set, and print mean MAE, naive MAE and skill per protocol, the rolling margin, and by how much the random skill is overstated.”
>
> **why the pro prompt works better** · 'Compare a random split and a time split fairly' typically gets an expanding `TimeSeriesSplit`, with each shuffled draw matched to its fold's sizes. Training size is therefore fair within a fold but grows across folds, and the averaged gap still mixes protocol with how much data each fit saw. A fixed sliding window appears only sometimes, and the splitter picks the origins. Copy-last-week on every test set and 20 shuffled draws usually come unasked, but nothing requires them, and the printed summary varies. The pro's `L, T = 700, 98`, rolling from starts 700 to 1092, makes every fit the same size and fixes the table this notebook discusses.
>
> ---
>
> **try** · set L = 200, keeping T = 98, and re-run. Both protocols get worse and the gap between them narrows. How much of the backtest's disadvantage was the protocol, and how much was training on a fifth of the data?

In [ ]:
from sklearn.model_selection import ShuffleSplit

L, T = 700, 98          # a fixed training size, and a quarter to score

# The rolling-origin backtest: five forward folds, each trained on the SAME
# number of days. TimeSeriesSplit's first fold trains on a fifth of the data, so
# comparing it with a random split confounds the protocol with the training-set
# size. Fixing the size removes that confound.
roll, roll_naive = [], []
for start in (700, 798, 896, 994, 1092):
    te  = np.arange(start, min(start + T, len(Xw)))
    trn = np.arange(start - L, start)
    fit_r = LinearRegression().fit(Xw[trn], yw[trn])
    roll.append(1e6 * np.abs(fit_r.predict(Xw[te]) - yw[te]).mean())
    roll_naive.append(1e6 * np.abs(Xw[te, -7] - yw[te]).mean())
ROLLING       = float(np.mean(roll))
ROLLING_NAIVE = float(np.mean(roll_naive))

matched, matched_naive = [], []
for a, b in ShuffleSplit(20, train_size=L, test_size=T,
                         random_state=RANDOM_STATE).split(Xw):
    fit_m = LinearRegression().fit(Xw[a], yw[a])
    matched.append(1e6 * np.abs(fit_m.predict(Xw[b]) - yw[b]).mean())
    matched_naive.append(1e6 * np.abs(Xw[b, -7] - yw[b]).mean())
MATCHED_RANDOM       = float(np.mean(matched))
MATCHED_RANDOM_NAIVE = float(np.mean(matched_naive))

print(f"train {L} days, test {T} days, in both protocols\n")
print(f"{'protocol':26s}{'model':>10s}{'naive':>10s}{'skill':>8s}")
for name, m, n in (("rolling origin, forward", ROLLING, ROLLING_NAIVE),
                   ("matched random, 20 draws", MATCHED_RANDOM,
                    MATCHED_RANDOM_NAIVE)):
    print(f"{name:26s}{m:>10,.0f}{n:>10,.0f}{m / n:>8.3f}")
print(f"\nrolling margin over copying last week  {1 - ROLLING / ROLLING_NAIVE:.1%}")
over = (ROLLING / ROLLING_NAIVE) / (MATCHED_RANDOM / MATCHED_RANDOM_NAIVE) - 1
print(f"the random split overstates the skill by {over:.1%}")
print("\nThe naive column needs no fitting, so it measures only how hard the")
print("rows are. The skill column is what makes the two protocols comparable.")

## 4 · Windows, for a network

Supervised learning wants rows. A series is not rows, so we cut it into
overlapping windows: 56 days in, the 57th day out.

Why 56? Eight whole weeks. Any multiple of seven lets a model line up the same
weekday; eight of them give it enough repetitions to average over. And why divide
by a million — because the numbers are around 600,000 and a network initialised
in the usual way expects inputs near 1.

> **Specification**
>
> **input** · a series as a tensor, and a window length
>
> **output** · a Dataset yielding (window, next step) pairs
>
> **constraint** · the target must be the step AFTER the window and never inside it — an off-by-one leaks one day and nothing later complains
>
> **check** · length is len(series) - window_length
>
> ---
>
> **novice prompt** · “make a sliding-window dataset for the RNN”
>
> **pro prompt** · “Write `TimeSeriesDataset(torch.utils.data.Dataset)` over a series tensor and a `window_length`: `__len__` returns `len(series) - window_length`, and `__getitem__(idx)` returns `series[idx:end]` and `series[end]`, with `end = idx + window_length` the first index after the window.”
>
> **why the pro prompt works better** · 'Make a sliding-window dataset for the RNN' is usually enough. Asked this way, a current assistant writes a map-style `Dataset` whose `__len__` is the series length minus the window. It slices exactly the window and takes the single next step as the target, often asserting the first target as well. But the request does not require any of that, so nothing guarantees that a target never slips inside its window, where a recurrent model would read it and the training loss would look superb. The pro makes both edges stated requirements: `__len__` as `len(series) - window_length`, and `end = idx + window_length` named as the first index after the window.
>
> ---
>
> **try** · return self.series[idx:end + 1] as the window. __len__ is unchanged and the target is now the last element of its own window. Which assert in the next cell catches it, and what does it print?

In [ ]:
class TimeSeriesDataset(torch.utils.data.Dataset):
    """Every window of `window_length` steps, and the step that follows it."""

    def __init__(self, series, window_length):
        self.series, self.window_length = series, window_length

    def __len__(self):
        return len(self.series) - self.window_length

    def __getitem__(self, idx):
        end = idx + self.window_length     # first index after the window
        return self.series[idx:end], self.series[end]

> **Specification**
>
> **input** · the tidied frame; the same pool and window as Section 3
>
> **output** · X of shape (rows, 56, 1) and y of shape (rows, 1)
>
> **constraint** · assert the shapes rather than printing them, so a wrong shape stops the notebook
>
> **check** · rows equals len(pool) - 56
>
> ---
>
> **novice prompt** · “build X and y and set up the train/test split”
>
> **pro prompt** · “Take `pool` as rail boardings from 2016-01 to 2019-05 with `WINDOW = 56`; stack windows of `pool / 1e6` into `X` (rows, 56, 1) and `y` (rows, 1), asserting both shapes. Split by time at `cut = int(0.8 * len(X))`. Compute copy-last-week MAE on the test rows and on all rows, and refit the forward linear model as `honest`.”
>
> **why the pro prompt works better** · 'build X and y and set up the train/test split' names neither the pool nor the bar. Asked this way, an assistant does split by time, but typically runs the data past 2019-05, so the test rows are days this notebook never scored. It computes no copy-last-week baseline on the test rows and no `honest` refit. It sometimes leaves `X` without its feature axis and only prints the shapes. The pro fixes `pool` from 2016-01 to 2019-05 and `cut = int(0.8 * len(X))`, asserts the (rows, 56, 1) shape, and scores the baseline on the rows the models are scored on.
>
> ---
>
> **try** · change WINDOW to 28 here and leave Section 3 alone. Both sections still run, both print an MAE, and the two are no longer comparable. Which printed line would have told you, and does any assert?

In [ ]:
# The same pool and the same window as Section 3, restated so that this section
# reads on its own: the networks must be scored on the same days as the baselines.
WINDOW = 56
pool = df["rail"]["2016-01":"2019-05"]
target = pool[WINDOW:]
assert len(target) == len(pool) - WINDOW

series = torch.tensor(pool.values / 1e6, dtype=torch.float32).unsqueeze(1)
dataset = TimeSeriesDataset(series, WINDOW)

X = torch.stack([w for w, _ in dataset])       # [rows, time, series]
y = torch.stack([t for _, t in dataset])

# The split, fixed here and used by every model below. By TIME, not at random:
# the windows overlap, so a shuffled split puts almost every test window's
# neighbours in the training set.
cut = int(len(X) * 0.8)

# The reference numbers for this split: the naive forecast every model has to
# beat, and the linear model under the honest forward split. These decide
# whether anything below is an improvement.
from sklearn.linear_model import LinearRegression

_naive = pool.shift(7)[WINDOW:]
_mask  = target.notna() & _naive.notna()

# The baseline has to be measured on the rows the models are scored on. Every
# model below is scored on the forward test slice only, so a baseline averaged
# over all 1,191 days would be a different set of days, and Section 3 showed
# why that comparison is not a comparison. The all-row figure is kept, but
# only to show how far apart the two are.
_test = _mask.copy()
_test.iloc[:cut] = False
NAIVE_ALL  = float((target[_mask] - _naive[_mask]).abs().mean())
NAIVE_MAE  = float((target[_test] - _naive[_test]).abs().mean())

_values = pool.values / 1e6
_Xl = np.stack([_values[i:i + WINDOW] for i in range(len(_values) - WINDOW)])
_yl = _values[WINDOW:]
honest = 1e6 * float(np.abs(
    LinearRegression().fit(_Xl[:cut], _yl[:cut]).predict(_Xl[cut:]) - _yl[cut:]).mean())

print(f"train {cut}, test {len(X) - cut}, split by time")
print(f"naive, test rows only   MAE {NAIVE_MAE:>10,.0f}")
print(f"naive, all 1,191 days   MAE {NAIVE_ALL:>10,.0f}  <- not comparable")
print(f"linear, forward split   MAE {honest:>10,.0f}")

assert X.shape == (len(pool) - WINDOW, WINDOW, 1), X.shape
assert y.shape == (len(pool) - WINDOW, 1), y.shape
print(f"X {tuple(X.shape)}   y {tuple(y.shape)}")
print("rows =", len(pool), "days -", WINDOW, "consumed by the first window")

## 5 · A recurrent network

The linear model treats lag 3 and lag 45 as two unrelated columns. A recurrent
layer instead reads the window one step at a time, carrying a hidden state:

$$\mathbf{h}_t = \tanh(W_x \mathbf{x}_t + W_h \mathbf{h}_{t-1} + \mathbf{b})$$

Same weights at every step. That is the parameter saving, and it is also the
assumption: *what a value means does not depend on where in the window it sits.*

> **Specification**
>
> **input** · a batch of windows, [batch, time, series]
>
> **output** · one number per window
>
> **constraint** · a linear head on the final step — the hidden state is bounded by tanh and ridership is not
>
> **check** · print the parameter count, split between recurrent and head
>
> ---
>
> **novice prompt** · “build a simple RNN model in PyTorch”
>
> **pro prompt** · “Write `SimpleRnn(nn.Module)` with `nn.RNN(1, 32, batch_first=True)` and an `nn.Linear(32, 1)` head applied to the output at the last time step. Instantiate it and print the parameter count, split between the recurrent layer and the head.”
>
> **why the pro prompt works better** · 'Build a simple RNN model' leaves the parameter count to chance. Asked this way, an assistant typically prints only the total, sometimes prints nothing, and never splits it by layer. It also tends to sprawl into a full training loop the request never asked for. A current assistant usually sets `batch_first=True` unasked and puts a linear head on the last step rather than returning the tanh-bounded state. The novice request requires neither, so nothing guarantees them. The pro states `nn.RNN(1, 32, batch_first=True)` and the `nn.Linear(32, 1)` head as requirements, and asks for the parameter count 'split between the recurrent layer and the head'.
>
> ---
>
> **try** · delete the head and return outputs[:, -1] directly. tanh bounds the hidden state to (-1, 1) and the scaled targets sit near 0.6, so it half works — which is worse than failing. Now try the same thing on the raw boardings, where it cannot.

In [ ]:
class SimpleRnn(nn.Module):
    def __init__(self, input_size=1, hidden_size=32, output_size=1):
        super().__init__()
        self.rnn = nn.RNN(input_size, hidden_size, batch_first=True)
        self.head = nn.Linear(hidden_size, output_size)

    def forward(self, X):                      # X: [batch, time, series]
        outputs, last_state = self.rnn(X)
        return self.head(outputs[:, -1])       # only the final step

model = SimpleRnn()
n_params = sum(p.numel() for p in model.parameters())
print(f"{n_params:,} parameters "
      f"({sum(p.numel() for p in model.rnn.parameters()):,} recurrent, "
      f"{sum(p.numel() for p in model.head.parameters()):,} in the head)")

**Why the head?** `nn.RNN` gives you a hidden state of 32 numbers, in [−1, 1]
because of the tanh. Ridership is not in [−1, 1]. The linear head is what turns a
state into a quantity, and leaving it out is the most common way this model is
got wrong.

### The training loop

Three choices worth stating:

* **Huber loss** with δ = 0.05 (that is 50,000 boardings). Quadratic near zero,
  linear far away — so a strike day steers the fit less than it would under MSE.
* **Shuffle the windows.** The *series* must not be shuffled; the *windows* must
  be, or every batch is one contiguous stretch and the gradients are correlated.
* **Split by time.** Same 80/20 cut as above, so the two models are comparable.

> **Specification**
>
> **input** · the training windows
>
> **output** · a trained network, with held-out MAE printed as it goes
>
> **constraint** · shuffle the WINDOWS but never the series; Huber loss so a strike day does not steer the fit; the same time split as the linear model
>
> **check** · held-out MAE should fall then flatten; if it rises, say so rather than picking the best epoch
>
> ---
>
> **novice prompt** · “train the RNN on the ridership data”
>
> **pro prompt** · “Train `SimpleRnn` on `X[:cut], y[:cut]` with a `DataLoader` of batch 32 that shuffles windows, `nn.HuberLoss(delta=0.05)`, and SGD at lr 0.05 with momentum 0.9, for 200 epochs. Every 20 epochs print held-out MAE on `X[cut:]` in boardings, and report the final epoch, not the best one seen.”
>
> **why the pro prompt works better** · 'Train the RNN on the ridership data' leaves the optimiser, schedule and loss scale to the assistant. Asked this way, an assistant typically reaches for Adam with its own learning rate, stops early and restores a best checkpoint, and never prints held-out MAE as it trains. Without the notebook's prose it sometimes writes `nn.HuberLoss()` with the default delta, which on targets in millions behaves like MSE. It usually splits chronologically and shuffles windows unasked, but the request does not require either. The pro's SGD at lr 0.05 with momentum 0.9 for 200 epochs, `delta=0.05`, and 'report the final epoch, not the best one seen' fix the run this notebook discusses.
>
> ---
>
> **try** · swap HuberLoss for MSELoss and re-run. The held-out MAE gets worse, and one strike day is the reason: squared error spends the model's capacity on the largest residual, which is the day nobody was staffing for anyway.

In [ ]:
from torch.utils.data import DataLoader, TensorDataset

train_set = TensorDataset(X[:cut], y[:cut])
loader = DataLoader(train_set, batch_size=32, shuffle=True)

model = SimpleRnn()
loss_fn = nn.HuberLoss(delta=0.05)
opt = torch.optim.SGD(model.parameters(), lr=0.05, momentum=0.9)

EPOCHS = 200
history = []
for epoch in range(EPOCHS):
    model.train()
    for Xb, yb in loader:
        opt.zero_grad()
        loss_fn(model(Xb), yb).backward()
        opt.step()
    if (epoch + 1) % 20 == 0 or epoch == 0:
        model.eval()
        with torch.no_grad():
            val = 1e6 * (model(X[cut:]) - y[cut:]).abs().mean().item()
        history.append((epoch + 1, val))
        print(f"epoch {epoch + 1:>3d}   held-out MAE {val:>10,.0f}")

### Which of these is "the" number?

This lecture quotes the **single forward hold-out**, because it is the protocol
that matches how the model would actually be used: fit once on everything up to
a date, forecast forward from there. Section 3 measured two others on the
same series — a random five-fold and a rolling-origin backtest — and it is
worth going back to that table before accepting this one.

The useful discipline is not picking the smallest number. It is **saying which
protocol produced the one you quote**, so that a reader can reproduce it and a
colleague can disagree with it. A margin without a protocol attached is not a
result.

## 6 · More series, not more layers

The honest margin is smaller, so the improvements have to be real.

Stacking three recurrent layers is the reflex, and on 1,191 windows it buys
nothing you can rely on: the ladder in Section 7 shows its ranking against one
layer turning on the seed. What actually helps is giving the model something it does not already have: bus
ridership, and **tomorrow's day type**, which is known in advance from a calendar
and is therefore not a leak.

> **Specification**
>
> **input** · rail, bus, and tomorrow's day type from the calendar
>
> **output** · a five-column frame, day type one-hot encoded
>
> **constraint** · shift(-1) on the CALENDAR is legitimate and shift(-1) on the target is a leak — the test is whether the value is knowable at prediction time
>
> **check** · assert the column names, so a silent change in encoding stops the notebook. For each shifted column ask: would I know this value at 6pm the day before? If yes it is a feature, if no it is the answer. That question is the whole of leak detection and it takes five seconds a column.
>
> ---
>
> **novice prompt** · “add bus ridership and the day type as features”
>
> **pro prompt** · “Build `mulvar` from `df[['rail', 'bus']] / 1e6` plus `next_day_type = df['day_type'].shift(-1)`, one-hot encode with `pd.get_dummies(dtype=float)`, and assert the columns are rail, bus, next_day_type_A, next_day_type_U, next_day_type_W in that order. Restrict to 2016-01 to 2019-05, drop NaN rows, and print the shape and columns.”
>
> **why the pro prompt works better** · 'add bus ridership and the day type as features' usually works: an assistant typically uses tomorrow's type via `shift(-1)`, keeps rail, divides by 1e6 and guards against NaN. The request does not demand the shift, though, so nothing guarantees it. What it does not reliably do is pin the layout. Columns come back as `next_A` and the like, the order of the three day types varies, and nothing asserts which column is rail. The pro's assert that the columns are exactly rail, bus, next_day_type_A, next_day_type_U, next_day_type_W, in that order, fixes the layout this notebook goes on to read.
>
> ---
>
> **try** · use shift(+1) rather than shift(-1) on day_type: yesterday's, not tomorrow's. The column-name assert still passes and the model still trains, but the feature is now useless rather than leaky. Which of those two mistakes would you notice sooner?

In [ ]:
mulvar = df[["rail", "bus"]] / 1e6
mulvar["next_day_type"] = df["day_type"].shift(-1)   # known in advance
mulvar = pd.get_dummies(mulvar, dtype=float)         # 5 columns

assert list(mulvar.columns) == ["rail", "bus", "next_day_type_A",
                                "next_day_type_U", "next_day_type_W"], \
    list(mulvar.columns)
mulvar = mulvar["2016-01":"2019-05"].dropna()
print(mulvar.shape, list(mulvar.columns))
mulvar.head(3)

**`shift(-1)` again — and this time it is legitimate.** A `shift(-1)` on the
*target* would be a leak. Here it is applied to the calendar, and
the difference is not the sign: it is whether the value would be available at
the moment of the forecast. Tomorrow's day type is on a wall planner. Tomorrow's
ridership is not.

State the rule you are using, every time, in one line: *would I know this number
when I have to make the prediction?*

## 7 · The ladder, under the protocol the deck reports

Everything above built one model at a time to explain a mechanism. This section
runs the whole comparison at once, under the protocol the slides quote — because
a table of six models is only a table if every row was produced the same way.

Two things are held fixed and stated. The **recipe** — optimiser, learning rate
and epoch count — is chosen for each architecture on a slice of the *training*
period, never on the test period; an architecture that loses only because it was
given the wrong learning rate has not been shown to lose. And the **split** is
by time: everything before 2019 trains, everything from 2019 is scored.

All six, in one cell. It is the lecture's result rather than a detour.

> **Specification · six architectures, one protocol**
>
> **input** · the univariate and multivariate window sets
>
> **output** · test MAE, train MAE and the chosen recipe for each of six models
>
> **constraint** · select the recipe on a held-out slice of the TRAINING period and then refit on all of it — selecting on the test period is the failure this whole part of the course is about, and it is one line away at every step
>
> **check** · print the recipe each model chose beside its score. If every architecture chose the same one, the selection step is doing nothing and should be removed rather than reported.
>
> ---
>
> **novice prompt** · “set up code to compare several recurrent architectures fairly”
>
> **pro prompt** · “Write `make_arrays(frame, w, target, horizon)` returning windows, targets and target dates; `build_model(kind, ...)` for linear, rnn, deep (3 layers), gru and lstm, each with a linear head on the last step; `RECIPES` of sgd (lr 0.05, 200 epochs) and adam (0.005, 120); and `fit_torch` with a fixed seed, Huber loss and a seeded DataLoader. Add `predict_torch` and an MAE in boardings.”
>
> **why the pro prompt works better** · 'Set up code to compare several recurrent architectures fairly' leaves the harness open. Asked this way, an assistant does not reliably offer every model more than one optimiser. It sometimes trains them all with a single SGD setting, so an architecture can lose for want of a learning rate. When it does search, it picks its own rates and caps epochs below 200. The ladder drifts too, sometimes lacking the linear baseline or the 3-layer model. A fixed seed, Huber loss and a seeded DataLoader usually appear unasked, but nothing requires them. The pro's two named `RECIPES`, sgd (0.05, 200 epochs) and adam (0.005, 120), offered to every model, fix the table this notebook discusses.
>
> ---
>
> **try** · fix every model to the sgd recipe and re-run. At least one architecture gets substantially worse, which is what the selection step exists to prevent.

In [ ]:
import time
from torch.utils.data import DataLoader, TensorDataset


def make_arrays(frame, w=WINDOW, target=0, horizon=1):
    """Windows of w rows, labelled with the next value of column `target`."""
    V = frame.values.astype(np.float32)
    n = len(V) - w - horizon + 1
    X = np.stack([V[i:i + w] for i in range(n)])
    y = np.stack([V[i + w:i + w + horizon, target] for i in range(n)])
    return X, y, frame.index[w:w + n]


def build_model(kind, input_size, output_size=1):
    class Rnn(nn.Module):
        def __init__(self, cell, layers=1, hidden=32):
            super().__init__()
            self.rnn = cell(input_size, hidden, num_layers=layers,
                            batch_first=True)
            self.head = nn.Linear(hidden, output_size)

        def forward(self, x):
            o, _ = self.rnn(x)
            return self.head(o[:, -1])

    if kind == "linear":
        return nn.Sequential(nn.Flatten(), nn.LazyLinear(output_size))
    return {"rnn":     lambda: Rnn(nn.RNN),
            "deep":    lambda: Rnn(nn.RNN, layers=3),
            "gru":     lambda: Rnn(nn.GRU),
            "lstm":    lambda: Rnn(nn.LSTM)}[kind]()


# optimiser, learning rate, epochs. Two recipes, and every architecture is
# offered both, so no model loses for want of a learning rate.
RECIPES = {"sgd": ("sgd", 0.05, 200), "adam": ("adam", 0.005, 120)}


def fit_torch(kind, Xtr, ytr, recipe, seed=RANDOM_STATE):
    torch.manual_seed(seed)
    model = build_model(kind, Xtr.shape[2], ytr.shape[1])
    name, lr, epochs = RECIPES[recipe]
    Xt, yt = torch.tensor(Xtr), torch.tensor(ytr)
    if kind == "linear":
        model(Xt[:2])                     # LazyLinear needs one pass to build
    opt = (torch.optim.Adam(model.parameters(), lr=lr) if name == "adam"
           else torch.optim.SGD(model.parameters(), lr=lr, momentum=0.9))
    loss_fn = nn.HuberLoss(delta=0.05)    # a strike day must not steer the fit
    loader = DataLoader(TensorDataset(Xt, yt), batch_size=32, shuffle=True,
                        generator=torch.Generator().manual_seed(seed))
    model.train()
    for _ in range(epochs):
        for xb, yb in loader:
            opt.zero_grad()
            loss_fn(model(xb), yb).backward()
            opt.step()
    return model.eval()


def predict_torch(model, X):
    with torch.no_grad():
        return model(torch.tensor(X)).numpy()


def mae_1e6(pred, truth):
    return 1e6 * float(np.abs(np.asarray(pred) - np.asarray(truth)).mean())

> **Specification · the selection step, and the two window sets**
>
> **input** · the tidied frame
>
> **output** · univariate rail windows, multivariate windows, and a recipe-selection function
>
> **constraint** · the selection slice comes out of the TRAINING period — nothing at or after the test cut is read here, and the function says so in its docstring
>
> **check** · print the multivariate columns. The next day's day-type is a legitimate feature because it is known in advance; the same column shifted the other way would be a leak.
>
> ---
>
> **novice prompt** · “pick the hyperparameters for each model without cheating”
>
> **pro prompt** · “With `CUT_SEL = '2018-07-01'` and `CUT_END = '2019-01-01'`, write `select_and_fit(kind, X, y, dates, seed)`: fit each recipe on dates before CUT_SEL, score on CUT_SEL to CUT_END, refit the winner on everything before CUT_END, touch nothing later. Print how many rows the ladder scores, and compute copy-last-week MAE, MAPE and mean level on them; build the univariate and multivariate window sets and print their columns.”
>
> **why the pro prompt works better** · 'Pick the hyperparameters for each model without cheating' is honoured where it is explicit: asked this way, an assistant usually validates inside the training period and never reads 2019, though nothing in the request checks it. The rest is not. It cuts by a fraction of rows rather than a date, so the validation window drifts. It often defers the refit to another cell. It never prints how many rows the ladder scores, typically reusing an earlier baseline or scoring one on validation rows, with no MAPE or mean level. The pro's `CUT_SEL = '2018-07-01'`, the refit before `CUT_END`, and copy-last-week on the ladder's own rows close each gap.
>
> ---
>
> **try** · move CUT_SEL to 2019-03-01, inside the test period. Every recipe is now selected on days the model is scored on, nothing raises, and every number in the ladder improves. That improvement is the failure, not the result.

In [ ]:
CUT_SEL, CUT_END = "2018-07-01", "2019-01-01"

# The ladder scores from CUT_END, which is a third set of days again -- so it
# needs its own baseline for the same reason.
_lad = _mask & (target.index >= CUT_END)
NAIVE_LADDER = float((target[_lad] - _naive[_lad]).abs().mean())
NAIVE_LADDER_MAPE = float((100 * (target[_lad] - _naive[_lad]).abs()
                           / target[_lad].abs()).mean())
LADDER_MEAN = float(target[_lad].mean())
print(f"ladder rows: {int(_lad.sum())} days from {CUT_END}")
print(f"copy last week, ladder rows only  MAE {NAIVE_LADDER:>10,.0f}"
      f"   MAPE {NAIVE_LADDER_MAPE:5.1f}%")
print(f"mean level on those same days     {LADDER_MEAN:>10,.0f}"
      f"   <- the denominator for any 'share of a day'")

def select_and_fit(kind, X, y, dates, seed=RANDOM_STATE):
    """Choose the recipe inside the training period, then refit on all of it.

    Nothing at or after CUT_END is touched here. That is the whole point: the
    selection slice comes out of the training period, not out of the test set.
    """
    sel, trn = dates < CUT_SEL, dates < CUT_END
    held = trn & ~sel
    scores = {}
    for recipe in RECIPES:
        m = fit_torch(kind, X[sel], y[sel], recipe, seed)
        scores[recipe] = mae_1e6(predict_torch(m, X[held])[:, 0], y[held, 0])
    best = min(scores, key=scores.get)
    return fit_torch(kind, X[trn], y[trn], best, seed), best, scores


POOL_FROM, POOL_TO = "2016-01", "2019-05"   # the same window as Section 3
rail_f = df["rail"][POOL_FROM:POOL_TO].to_frame() / 1e6
mul_f  = df[["rail", "bus"]][POOL_FROM:POOL_TO] / 1e6
mul_f["next_day_type"] = df["day_type"].shift(-1)[POOL_FROM:POOL_TO]
mul_f = pd.get_dummies(mul_f, dtype=float)

Xu, yu, du = make_arrays(rail_f)
Xm, ym, dm = make_arrays(mul_f)
print(f"univariate   {Xu.shape}   multivariate {Xm.shape}")
print(f"multivariate columns: {list(mul_f.columns)}")

> **Specification · run the ladder**
>
> **input** · six (architecture, window set) pairs
>
> **output** · one row each: test MAE, train MAE, chosen recipe, both recipes' selection scores, wall clock; then each row's test MAPE; then the two univariate RNNs under three more seeds
>
> **constraint** · report the TRAIN MAE beside the test MAE — without it, a model that is worse on both is indistinguishable from one that overfits, and those have opposite fixes
>
> **check** · the naive baseline goes in the same table. Six architectures that all fail to beat copying last week is a finding, and it is one the table has to be able to show.
>
> ---
>
> **novice prompt** · “train all six models and compare them”
>
> **pro prompt** · “Call `select_and_fit` for linear, simple RNN and 3-layer RNN on the univariate windows, and simple RNN, GRU and LSTM on the multivariate ones. Print test MAE (dates from CUT_END), train MAE, chosen recipe, both recipes' selection scores and seconds per row, then `NAIVE_LADDER` for copy-last-week in the same table, each row's test MAPE beside `NAIVE_LADDER_MAPE`, and the simple RNN's selection margin. Then re-run `select_and_fit` for the simple and the deep RNN with seeds 0, 1 and 2, and print each seed's test MAE, train MAE and recipe for both beside seed 42's.”
>
> **why the pro prompt works better** · 'Train all six models and compare them' gets recipe selection, test MAE, the chosen recipe, timing and MAPE, but a current assistant typically never computes train MAE, so underfitting and overfitting look identical though they need opposite fixes. It also typically feeds GRU and LSTM univariate windows, leaving only the simple RNN on the multivariate frame, which changes what the table can conclude. Copy-last-week appears as a separate line rather than a row, and the simple RNN's selection margin is not reliably flagged, and nothing re-runs a comparison under a second seed. The pro's input split by architecture, train MAE beside each test MAE, `NAIVE_LADDER` in the same table and the seed re-runs fix the experiment and make each row diagnosable.
>
> ---
>
> **try** · read the train MAE column on its own. Which rows are worse on both, and which are far better on train than on test? Those two failures have opposite fixes, and the test column alone cannot tell them apart.

In [ ]:
ladder = [("linear", Xu, yu, du, "Linear, 56 lags"),
          ("rnn",    Xu, yu, du, "Simple RNN, 32 units"),
          ("deep",   Xu, yu, du, "Deep RNN, 3 layers"),
          ("rnn",    Xm, ym, dm, "Simple RNN, multivariate"),
          ("gru",    Xm, ym, dm, "GRU, multivariate"),
          ("lstm",   Xm, ym, dm, "LSTM, multivariate")]

print(f"{'model':28s}{'test MAE':>10s}{'train MAE':>11s}{'recipe':>8s}"
      f"{'sel sgd':>10s}{'sel adam':>10s}")
rows, selection, test_mape = [], {}, {}
for kind, X, y, dates, label in ladder:
    t0 = time.perf_counter()
    model, best, scores = select_and_fit(kind, X, y, dates)
    te, tr = dates >= CUT_END, dates < CUT_END
    pred_te = predict_torch(model, X[te])[:, 0]
    test_mae  = mae_1e6(pred_te, y[te, 0])
    test_mape[label] = float(100 * (np.abs(pred_te - y[te, 0])
                                    / np.abs(y[te, 0])).mean())
    train_mae = mae_1e6(predict_torch(model, X[tr])[:, 0], y[tr, 0])
    rows.append((label, test_mae, train_mae, best))
    selection[label] = scores
    print(f"{label:28s}{test_mae:>10,.0f}{train_mae:>11,.0f}{best:>8s}"
          f"{scores['sgd']:>10,.0f}{scores['adam']:>10,.0f}"
          f"   ({time.perf_counter() - t0:.0f}s)")

print(f"\n{'copy last week':28s}{NAIVE_LADDER:>10,.0f}")

# the same test days as a percentage of each day: the share a planner reads
print(f"\n{'test MAPE, same days':28s}")
for label, m in test_mape.items():
    print(f"{label:28s}{m:>9.1f}%")
print(f"{'copy last week':28s}{NAIVE_LADDER_MAPE:>9.1f}%")

# The two univariate RNNs differ in depth -- and, possibly, in recipe. Print
# how close the simple RNN's recipe choice was, because the ranking rests on it.
sel_rnn = selection["Simple RNN, 32 units"]
sel_gap = abs(sel_rnn["sgd"] - sel_rnn["adam"])
print(f"\nsimple RNN's selection: sgd {sel_rnn['sgd']:,.0f}"
      f" vs adam {sel_rnn['adam']:,.0f}  -> {min(sel_rnn, key=sel_rnn.get)}"
      f" by {sel_gap:,.0f} ({sel_gap / max(sel_rnn.values()):.1%})")

# One seed is one draw. Every fit above is seeded, so this table reproduces on
# one machine; re-run the two univariate RNNs -- selection, refit and all --
# under three more seeds, changing nothing else. (A few minutes.)
te, tr = du >= CUT_END, du < CUT_END
print(f"\n{'seed':>4s}{'simple RNN':>12s}{'train':>9s}{'recipe':>7s}"
      f"{'deep RNN':>10s}{'train':>9s}{'recipe':>7s}   shallower wins?")
for seed in (RANDOM_STATE, 0, 1, 2):
    line = []
    for kind, label in (("rnn", "Simple RNN, 32 units"), ("deep", "Deep RNN, 3 layers")):
        if seed == RANDOM_STATE:                    # the ladder's own rows
            line.append([r for r in rows if r[0] == label][0][1:])
            continue
        m, best, _ = select_and_fit(kind, Xu, yu, du, seed)
        line.append((mae_1e6(predict_torch(m, Xu[te])[:, 0], yu[te, 0]),
                     mae_1e6(predict_torch(m, Xu[tr])[:, 0], yu[tr, 0]), best))
    (s_te, s_tr, s_r), (d_te, d_tr, d_r) = line
    print(f"{seed:>4d}{s_te:>12,.0f}{s_tr:>9,.0f}{s_r:>7s}"
          f"{d_te:>10,.0f}{d_tr:>9,.0f}{d_r:>7s}   {'yes' if s_te < d_te else 'no'}")

> **Specification · the recurrent model under a random split**
>
> **input** · the univariate windows, five shuffled folds
>
> **output** · the RNN's MAE under the protocol Section 3 showed was wrong
>
> **constraint** · the same architecture and the same recipe as the ladder's row — only the splitter changes, or the comparison is not about the splitter
>
> **check** · compare it with the ladder's forward-split row. The raw gap is the protocol PLUS the rows -- the two protocols score different days, and the forward ones are harder. Put each number beside its own naive baseline: the skill ratio is what isolates the protocol, and it is still larger than every architectural difference in the table above.
>
> ---
>
> **novice prompt** · “what if I cross-validated the RNN with k-fold?”
>
> **pro prompt** · “Run `KFold(5, shuffle=True, random_state=RANDOM_STATE)` on `Xu`, fitting `fit_torch('rnn', ...)` in each fold with the recipe the ladder chose for its simple-RNN row, and average the MAE; score copy-last-week on the same folds. Print MAE, naive and skill for the random folds and for the ladder's simple-RNN forward row, then the raw difference and the change in skill.”
>
> **why the pro prompt works better** · 'Cross-validate the RNN with k-fold' is enough for a current assistant to shuffle 5 folds with the seed, reuse the ladder's recipe through `fit_torch('rnn', ...)` and score copy-last-week on the same rows. What it does not reliably do is set the two protocols against each other. It prints a ratio for each but rarely the forward-minus-random gap, never the change in skill beside it, and sometimes tells the reader to read the raw gap. It also typically limits the folds to the training period, not all of `Xu`. The pro's 'the raw difference and the change in skill' is what separates the protocol from the easier rows.
>
> ---
>
> **try** · run the same five folds with KFold(5, shuffle=False). It is still not a forward split — fold 1 trains on the future of fold 5 — but it does keep neighbouring windows together. Where does its number fall between the two printed here, and which of the two leaks does that isolate?

In [ ]:
from sklearn.model_selection import KFold

# The ladder row's own recipe, not a re-selection: only the splitter changes.
_, forward, _, rnn_recipe = [r for r in rows if r[0] == "Simple RNN, 32 units"][0]

folds = []
for tr_i, te_i in KFold(5, shuffle=True, random_state=RANDOM_STATE).split(Xu):
    m = fit_torch("rnn", Xu[tr_i], yu[tr_i], rnn_recipe)
    folds.append(mae_1e6(predict_torch(m, Xu[te_i])[:, 0], yu[te_i, 0]))
RNN_RANDOM_CV = float(np.mean(folds))

# The two protocols score DIFFERENT rows, so the raw difference mixes the
# protocol with how hard those rows are. Each number needs its own baseline,
# and the skill ratio -- model over naive on the same rows -- is what isolates
# the protocol. This is Section 3's own warning applied to our own comparison.
naive_random = float(np.mean([
    mae_1e6(Xu[te_i][:, -7, 0], yu[te_i, 0])
    for _, te_i in KFold(5, shuffle=True, random_state=RANDOM_STATE).split(Xu)]))

print(f"recipe, as the ladder chose it: {rnn_recipe}\n")
print(f"{'':28s}{'MAE':>10s}{'naive':>10s}{'skill':>8s}")
print(f"{'random 5-fold':28s}{RNN_RANDOM_CV:>10,.0f}{naive_random:>10,.0f}"
      f"{RNN_RANDOM_CV / naive_random:>8.3f}")
print(f"{'forward split':28s}{forward:>10,.0f}{NAIVE_LADDER:>10,.0f}"
      f"{forward / NAIVE_LADDER:>8.3f}")
print()
print(f"raw difference  {forward - RNN_RANDOM_CV:>10,.0f}  <- protocol AND rows")
print(f"skill worsens by{forward / NAIVE_LADDER - RNN_RANDOM_CV / naive_random:>10.3f}"
      f"  <- the protocol alone")

### Several steps ahead

One output was a choice. A system that plans staffing needs a fortnight, not a
day — so give the model fourteen outputs and read the error at each horizon.

> **Specification · fourteen horizons from one model**
>
> **input** · the multivariate windows, labelled with the next fourteen days
>
> **output** · MAE at each horizon, beside a naive forecast at the same horizon
>
> **constraint** · the naive reference has to be honest at every horizon — the last value of the SAME weekday still inside the window, which is a different window position for each h
>
> **check** · the error should rise with the horizon and then flatten. If it is flat from the start, the model is predicting the weekly pattern and nothing else.
>
> ---
>
> **novice prompt** · “predict two weeks ahead instead of one day”
>
> **pro prompt** · “Build multivariate windows labelled with the next 14 days via `make_arrays(mul_f, horizon=14)`, select and fit a GRU with `select_and_fit`, and on dates from CUT_END print MAE at each horizon beside a naive forecast copying the latest same-weekday value inside the window: position 48+h for h ≤ 7 and 41+h for h ≤ 14.”
>
> **why the pro prompt works better** · 'Predict two weeks ahead instead of one day' names no model and no input. Asked this way, an assistant does not reliably fit a GRU; it typically fits a linear model, a simple RNN or an LSTM instead. Without the notebook's prose it may also fall back to the univariate series and drop `mul_f`. It usually builds an honest naive forecast unasked, repeating the last observed week and printing it beside each model at every horizon. The request does not require that, so nothing guarantees it. The pro fixes `make_arrays(mul_f, horizon=14)` and a GRU through `select_and_fit`, and requires the same-weekday naive at position 48+h or 41+h.
>
> ---
>
> **try** · replace naive_pos with a constant 55 — yesterday, at every horizon. The naive column stops rising with h and the GRU suddenly looks excellent at t+14. A baseline that does not get harder with the horizon is not a baseline for that horizon.

In [ ]:
Xa, ya, da = make_arrays(mul_f, horizon=14)
model, best, _ = select_and_fit("gru", Xa, ya, da)
te = da >= CUT_END
P = predict_torch(model, Xa[te])

# Window position 55 is day t, so the same weekday before t+h sits at 48+h for
# h <= 7 and at 41+h for h <= 14.
naive_pos = [48 + h if h <= 7 else 41 + h for h in range(1, 15)]
ahead = [mae_1e6(P[:, h], ya[te, h]) for h in range(14)]
ahead_naive = [mae_1e6(Xa[te, naive_pos[h], 0], ya[te, h]) for h in range(14)]

print(f"recipe chosen: {best}")
print(f"{'horizon':>8s}{'GRU':>10s}{'naive':>10s}")
for h in (0, 6, 13):
    print(f"{'t+' + str(h + 1):>8s}{ahead[h]:>10,.0f}{ahead_naive[h]:>10,.0f}")
print(f"\nt+1 {ahead[0]:,.0f} rises to t+14 {ahead[13]:,.0f}")

### Regime change

A regime change is not a leak — it is a fact about the world. Fit on everything
to the end of 2019, forecast the spring of 2020, and put copy-last-week on the
same days beside it.

> **Specification · what an honest protocol says when the world changes**
>
> **input** · the series extended through 2021, and the spring of 2020
>
> **output** · a forward fit scored on March to June 2020, beside copy-last-week on the same days and the level of those days
>
> **constraint** · score the naive forecast on the same 2020 rows — it needs no fitting, so it separates 'the model was wrong' from 'the days were unlike anything before them'
>
> **check** · report the level as well as the error. A model can be wrong by 84,000 on days whose mean is 141,000, and that ratio is the whole story.
>
> ---
>
> **novice prompt** · “how do the models do on 2020?”
>
> **pro prompt** · “Build 56-lag rail windows from 2016-01 to 2021-11. Fit `LinearRegression` on targets before 2020 and score MAE on March to June 2020, beside copy-last-week on the same rows, how many days they are, and the mean level on those days, all in boardings.”
>
> **why the pro prompt works better** · 'How do the models do on 2020?' leaves the scoring window open, and asked this way an assistant typically scores March to May or its own split of the year, so the March to June figures never appear. It refits whatever models it already has rather than a plain `LinearRegression`, and it does not reliably print the day count or mean level beside the error. It usually keeps the split forward and adds copy-last-week unasked, though the request requires neither. The pro fixes the window at March to June 2020 and prints the day count and mean level beside copy-last-week, so an error of 84,000 can be read against days averaging 141,000.
>
> ---
>
> **try** · score the same forward fit on March to June 2019 instead of 2020, changing nothing else. The MAE falls by most of the gap. Neither the model nor the protocol changed between those two runs.

In [ ]:
pool21 = df["rail"]["2016-01":"2021-11"]
v21 = pool21.values / 1e6
X21 = np.stack([v21[i:i + WINDOW] for i in range(len(v21) - WINDOW)])
y21 = v21[WINDOW:]
d21 = pool21.index[WINDOW:]

tr21 = d21 < "2020-01-01"
te21 = (d21 >= "2020-03-01") & (d21 < "2020-07-01")
fit21 = LinearRegression().fit(X21[tr21], y21[tr21])
mae21 = mae_1e6(fit21.predict(X21[te21]), y21[te21])
naive21 = mae_1e6(X21[te21, -7], y21[te21])

print(f"forward fit, scored on 2020      MAE {mae21:>10,.0f}")
print(f"copy last week, same 2020 rows   MAE {naive21:>10,.0f}   ({int(te21.sum())} days)")
print(f"the actual level on those days       {1e6 * y21[te21].mean():>10,.0f}")
print(f"\nthe model is {mae21 / naive21:.1f} times worse than copying last week")

**Read the train column beside the test column — and then the seed table.**
Which of the two univariate RNNs wins is not settled by the ladder. Under seed
42 the simple RNN chose sgd on two selection scores a few per cent apart (the
line under the ladder prints them), and the deep RNN came out worse on both test
and train. Change only the seed and the order flips, and back again; some seeds
even choose a recipe whose refit fails on the training rows themselves. Every
fit is seeded, so each line reproduces exactly on one machine, but floating-point
rounding differs between processors and library versions, and these fits
amplify it: on another machine seed 42 can land on a different line of this
table. A ranking whose gap is smaller than its spread over seeds is not a
finding about depth: report it as unresolved.

The multivariate rows are the interesting ones: adding bus ridership and the
*next* day's day-type is worth more than any architectural change on this
series. The gates help, but the extra series helps more — and knowing which is
which is why every row here changed exactly one thing.

## 8 · Where we are

- A recurrent cell replaces a fixed window with a state, and unrolling shows it
  is a deep network with shared weights.
- Backpropagation through time is a deep stack's vanishing or exploding
  gradient with one aggravation:
  the same matrix multiplies the gradient at every step, so its powers vanish
  or explode.
- A gate is a path along which the gradient is *added to* rather than
  multiplied. Everything else about LSTM and GRU is arrangement.
- Copying last week is the number to beat, and a shuffled split overstates a
  forecast's skill; every model here was scored on one forward hold-out, so the
  rows can be read against one another.

**Before the next lecture:** run this notebook, then change the seed in the
setup cell to one the seed table does not show and re-run the ladder. Watch the
simple and the deep RNN: if their order flips again, you have reproduced the
point about a ranking that rests on one seed.